# 08 — Clustering no supervisado (K-Means, GMM, Aglomerativo y HDBSCAN)

**Pregunta:** ¿los agarres aparecen solos, sin usar las etiquetas?

Se le dan al algoritmo las filas **sin decirle qué agarre es cada una**, se le pide que las separe en grupos
por parecido, y al final se compara si esos grupos coinciden con los agarres reales.

El clustering se hace **directamente sobre las columnas del dataset** (sin PCA).

### Métricas de calidad (las únicas usadas en el notebook)

| Métrica | Qué mide | Mejor valor |
|---|---|---|
| **Silhouette** | Cohesión y separación de los grupos | **Alto** (de −1 a 1) |
| **Davies–Bouldin** | Dispersión interna y separación relativa | **Bajo** (0 = perfecto) |
| **ARI** (Adjusted Rand Index) | Coincidencia con los agarres reales | **Alto** (0 = azar, 1 = perfecto) |

Las figuras **no se muestran en el notebook**: se guardan como PDF en `figures/clustering/<dataset>/`
(con la fecha de generación en la esquina), igual que en el notebook 07. Cada método genera su
tabla de contingencia (en **proporciones**, no conteos), su vista **2D** y su vista **3D**, y al final
hay una **tabla-resumen** (`resumen_metricas.pdf`) con las tres métricas y el número de grupos
elegido por cada método.

Cada sección de cálculo está implementada como **función** (`run_kmeans`, `run_gmm`, …); las celdas
solo las llaman y muestran resultados.

### Cómo reutilizarlo con otro dataset
Cambia `DATA_PATH` (y opcionalmente `FEATURE_COLS`) en la primera celda de código.
Si `FEATURE_COLS = None`, las columnas se detectan automáticamente: todas las numéricas
que no sean metadatos (`stimulus`, `subject`, `origin_db`, ventanas, etc.).

| Sección | Contenido |
|---|---|
| 1 | Carga (con detección automática de features) |
| 2 | K-Means con k = número de agarres: métricas, tabla % y vistas 2D/3D |
| 3 | K-Means con otros valores de k |
| 4 | GMM: barrido de n_components con BIC/AIC |
| 5 | Aglomerativo: comparación de linkages |
| 6 | HDBSCAN con min_samples explícito |
| 7 | ¿Es mejor que el azar? (permutación) |
| 8 | Resumen (tabla-resumen de métricas en PDF) |
| 9 | Interpretación |

In [1]:
from _provenance import Sello
sello = Sello('08_clustering.ipynb')

In [2]:
# Cambia SOLO estas dos líneas para probar otro dataset.
DATA_PATH = '../data/prueba/df_flat_wnd_mean.csv'
FEATURE_COLS = None   # None = detecta automáticamente las columnas numéricas no-meta

In [3]:
# ---------------- Configuración (normalmente no se toca) ----------------
LABEL_COL, SUBJECT_COL, DB_COL = 'stimulus', 'subject', 'origin_db'
META_COLS = {LABEL_COL, SUBJECT_COL, DB_COL, 'repetition', 'window',
             'window_index', 'window_size_samples', 'window_size_ms'}
REST = 0
K_RANGE = range(2, 13)                # valores de k a probar en la sección 3
GMM_N_RANGE = range(2, 13)            # n_components a probar en la sección 4
# 'diag': con 150 columnas y ~600 filas, 'full' tiene >11k parámetros por componente y el BIC
# queda dominado por la penalización (siempre elige el mínimo n). 'diag' es estimable.
GMM_COVARIANCE = 'diag'
AGG_LINKAGES = ['ward', 'complete', 'average']   # linkages a probar (sección 5)
HDB_MIN_SIZES = [15, 20, 30, 40]      # tamaños mínimos de grupo (sección 6)
HDB_MIN_SAMPLES = [1, 3, 5, 10]       # min_samples explícitos (sección 6)
HDB_MAX_RUIDO = 60                    # % máximo de ruido para aceptar una combinación de HDBSCAN
N_INIT = 50                           # reinicios de K-Means (sección 2 y barrido, iguales)
N_PERM = 1000                         # repeticiones de la prueba de azar
RANDOM_STATE = 42
EMBED_2D = 'tsne'                     # cómo dibujar en 2D: 'tsne' | 'umap'
# -------------------------------------------------------------------------

import os
import warnings
from datetime import datetime
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib import colormaps
from matplotlib.patches import Rectangle
import seaborn as sns

from sklearn.cluster import KMeans, HDBSCAN, AgglomerativeClustering
from sklearn.manifold import TSNE
from sklearn.mixture import GaussianMixture
from sklearn.metrics import (adjusted_rand_score, davies_bouldin_score,
                             silhouette_score)

warnings.filterwarnings('ignore')
sns.set_style('whitegrid')
plt.rcParams.update({'figure.dpi': 120, 'font.size': 11})

# Fecha de generación de las figuras (sello en la esquina de cada PDF)
GEN_DATE = datetime.now().strftime('%Y-%m-%d %H:%M')


def style_ax(ax):
    ax.set_facecolor('#FFFFFF')
    ax.spines[['top', 'right']].set_visible(False)
    ax.spines[['left', 'bottom']].set_color('#CCCCCC')
    ax.tick_params(colors='#444444', labelsize=8)


def savefig(fig, name):
    """Guarda la figura en PDF dentro de SAVE_DIR y libera la memoria (sin plt.show)."""
    fig.text(0.99, 0.01, f'Generado: {GEN_DATE}', ha='right', va='bottom',
             fontsize=7, color='#666666', transform=fig.transFigure)
    fig.savefig(f'{SAVE_DIR}/{name}.pdf', dpi=300, bbox_inches='tight')
    plt.close(fig)

In [4]:
# ---------- Funciones de análisis (una por sección) ----------

def cargar_datos(data_path, feature_cols=None, rest=REST, meta_cols=META_COLS,
                 label_col=LABEL_COL, subject_col=SUBJECT_COL, db_col=DB_COL):
    """Lee el CSV, detecta features, quita el reposo y arma X/y/groups."""
    df = pd.read_csv(data_path)
    missing = [c for c in [label_col, subject_col, db_col] if c not in df.columns]
    if missing:
        raise KeyError(f'Faltan columnas en el CSV ({len(missing)}): {missing}')

    if feature_cols is None:
        feature_cols = [c for c in df.columns
                        if c not in meta_cols and pd.api.types.is_numeric_dtype(df[c])]
        if not feature_cols:
            raise KeyError('No se detectó ninguna columna numérica de features.')
    else:
        missing_f = [c for c in feature_cols if c not in df.columns]
        if missing_f:
            raise KeyError(f'feature_cols no está en el CSV ({len(missing_f)}): {missing_f[:10]}')

    df = df[df[label_col] != rest].copy().reset_index(drop=True)
    df['group'] = df[subject_col].astype(str) + '_' + df[db_col].astype(str)

    X = df[feature_cols].values.astype(float)
    y = df[label_col].values
    groups = df['group'].values
    classes = sorted(np.unique(y).tolist())
    return df, X, y, groups, classes


def evaluate(X, labels, y):
    """Las 3 métricas: Silhouette y Davies-Bouldin (calidad interna, sin ruido −1)
    + ARI contra los agarres reales. Si hay menos de 2 grupos, Silhouette/DB = NaN."""
    mask = labels >= 0
    if len(set(labels[mask])) > 1:
        sil = silhouette_score(X[mask], labels[mask])
        db = davies_bouldin_score(X[mask], labels[mask])
    else:
        sil = db = np.nan
    return {'silhouette': sil, 'davies_bouldin': db,
            'ARI_agarre': adjusted_rand_score(y, labels)}


def run_kmeans(X, y, k, n_init=N_INIT, seed=RANDOM_STATE):
    labels = KMeans(n_clusters=k, n_init=n_init, random_state=seed).fit_predict(X)
    res = {'k': k, **evaluate(X, labels, y)}
    return labels, res


def barrido_kmeans(X, y, ks=K_RANGE, n_init=N_INIT, seed=RANDOM_STATE):
    rows = []
    for k in ks:
        lab = KMeans(n_clusters=k, n_init=n_init, random_state=seed).fit_predict(X)
        rows.append({'k': k, **evaluate(X, lab, y)})
    df = pd.DataFrame(rows)
    best = int(df.loc[df['silhouette'].idxmax(), 'k'])
    return df, best


def run_gmm(X, y, n_range=GMM_N_RANGE, covariance=GMM_COVARIANCE, seed=RANDOM_STATE):
    """Barrido de n_components con BIC/AIC; el modelo final es el del barrido con mejor BIC."""
    rows, all_labels = [], {}
    for n in n_range:
        g = GaussianMixture(n_components=n, covariance_type=covariance,
                            n_init=10, random_state=seed).fit(X)
        lab = g.predict(X)
        all_labels[n] = lab
        rows.append({'n_components': n, 'BIC': g.bic(X), 'AIC': g.aic(X),
                     **evaluate(X, lab, y)})
    df = pd.DataFrame(rows)
    best_n = int(df.loc[df['BIC'].idxmin(), 'n_components'])

    labels = all_labels[best_n]
    res = {'n_components': best_n,
           **df.loc[df['n_components'] == best_n,
                    ['silhouette', 'davies_bouldin', 'ARI_agarre']].iloc[0].to_dict()}
    return df, best_n, labels, res


def run_aglomerativo(X, y, k, linkages=AGG_LINKAGES):
    labels, rows = {}, []
    for link in linkages:
        lab = AgglomerativeClustering(n_clusters=k, linkage=link).fit_predict(X)
        labels[link] = lab
        rows.append({'linkage': link, **evaluate(X, lab, y)})
    df = pd.DataFrame(rows).set_index('linkage')
    # Selección sin etiquetas: mejor silhouette (el ARI queda solo como evaluación)
    best = df['silhouette'].idxmax()
    return df, labels, best


def run_hdbscan(X, y, min_sizes=HDB_MIN_SIZES, min_samples=HDB_MIN_SAMPLES,
                max_ruido=HDB_MAX_RUIDO):
    """Grid min_cluster_size × min_samples (explícito). Mejor combinación por silhouette
    (sin etiquetas) entre las que tienen >1 grupo y ruido <= max_ruido %."""
    labels, rows = {}, []
    for mcs in min_sizes:
        for ms in min_samples:
            lab = HDBSCAN(min_cluster_size=mcs, min_samples=ms).fit_predict(X)
            labels[(mcs, ms)] = lab
            rows.append({'min_cluster_size': mcs, 'min_samples': ms,
                         'n_grupos': len(set(lab[lab >= 0])),
                         '%_ruido': 100 * (lab < 0).mean(),
                         **evaluate(X, lab, y)})
    df = pd.DataFrame(rows).set_index(['min_cluster_size', 'min_samples'])
    valid = df[(df['n_grupos'] > 1) & (df['%_ruido'] <= max_ruido)]
    best = valid['silhouette'].idxmax() if len(valid) else None
    return df, labels, best


def prueba_azar(y, tests, n_perm=N_PERM, seed=RANDOM_STATE):
    """tests: [{'nombre', 'labels', 'ari', 'color'}]. Distribución nula por algoritmo."""
    rng = np.random.default_rng(seed)
    nulls, p_vals = {}, {}
    for t in tests:
        null = np.array([adjusted_rand_score(rng.permutation(y), t['labels'])
                         for _ in range(n_perm)])
        nulls[t['nombre']] = null
        p_vals[t['nombre']] = (np.sum(null >= t['ari']) + 1) / (n_perm + 1)
    return nulls, p_vals

## 1. Carga

El dataset ya viene transformado: solo se carga y se quita el reposo. **No se re-escala**: las
columnas tienen escalas parecidas (desviación ≈ 1–2), pero no están estandarizadas.
Si `FEATURE_COLS = None` se detectan automáticamente las columnas numéricas que no son metadatos.
El sujeto se identifica con el ID compuesto `subject + origin_db`.

In [5]:
df, X, y, groups, CLASSES = cargar_datos(DATA_PATH, FEATURE_COLS)
N_CLASSES = len(CLASSES)
CLASS_COLORS = dict(zip(CLASSES, sns.color_palette('tab10', N_CLASSES)))

N_FEATURES = len(FEATURE_COLS) if FEATURE_COLS is not None else X.shape[1]
DATASET_TAG = f'{Path(DATA_PATH).stem}__{N_FEATURES}f'
SAVE_DIR = f'../figures/clustering/{DATASET_TAG}'
os.makedirs(SAVE_DIR, exist_ok=True)

print(f'Dataset: {DATA_PATH} ({N_FEATURES} columnas)')
print(f'Figuras: {SAVE_DIR}')
print(f'Filas: {len(df)} | agarres ({N_CLASSES}): {CLASSES} | sujetos: {len(np.unique(groups))}')

Dataset: ../data/prueba/df_flat_wnd_mean.csv (150 columnas)
Figuras: ../figures/clustering/df_flat_wnd_mean__150f
Filas: 609 | agarres (7): [22, 25, 27, 30, 31, 37, 40] | sujetos: 87


In [6]:
# ---------- Funciones de gráficas (ven los datos como globals del dataset) ----------

def plot_contingency(labels, title, fname):
    ct = pd.crosstab(pd.Series(labels, name='Grupo encontrado'),
                     pd.Series(y, name='Agarre real'))
    ct = ct.reindex(columns=CLASSES, fill_value=0)
    ct_norm = ct.div(ct.sum(axis=0), axis=1)
    # 2 decimales: cada columna suma 100.00 (redondeo a enteros daba 97–101%)
    ct_pct = (ct_norm * 100).round(2)

    fig, ax = plt.subplots(figsize=(1.3 * N_CLASSES + 2, 0.55 * len(ct) + 1.8), dpi=300)
    fig.patch.set_facecolor('#F7F9FC')
    sns.heatmap(ct_pct, annot=ct_pct, fmt='.2f', cmap='Blues', vmin=0, vmax=100,
                linewidths=0.5, linecolor='#FFFFFF', annot_kws={'fontsize': 8},
                cbar_kws={'label': 'Proporción del agarre (%)'}, ax=ax)
    ax.set_title(title, fontweight='bold', fontsize=10)
    plt.tight_layout()
    savefig(fig, fname)

def proyectar(X, n_components, embed=EMBED_2D, seed=RANDOM_STATE):
    """Embedding 2D o 3D (t-SNE o UMAP); se calcula una vez y se reutiliza."""
    if embed == 'umap':
        import umap
        coords = umap.UMAP(n_components=n_components, n_neighbors=30, min_dist=0.1,
                           random_state=seed).fit_transform(X)
        return coords, 'UMAP'
    coords = TSNE(n_components=n_components, perplexity=30, init='pca',
                  random_state=seed).fit_transform(X)
    return coords, 't-SNE'


def plot_clusters_2d(labels, title, fname):
    # Izquierda: grupos encontrados. Derecha: agarres reales. Mismas coordenadas.
    fig, axes = plt.subplots(1, 2, figsize=(15, 6.2), dpi=300, sharex=True, sharey=True)
    fig.patch.set_facecolor('#F7F9FC')

    ax = axes[0]
    found = [c for c in sorted(set(labels)) if c >= 0]
    pal = sns.color_palette('Set2', max(1, len(found)))
    if (labels < 0).any():
        m = labels < 0
        ax.scatter(E2D[m, 0], E2D[m, 1], s=14, color='#BDBDBD', alpha=0.5, label='Ruido (−1)')
    for c, col in zip(found, pal):
        m = labels == c
        ax.scatter(E2D[m, 0], E2D[m, 1], s=28, color=col, alpha=0.85,
                   edgecolor='white', linewidth=0.4, label=f'Grupo {c}')
        ax.text(np.median(E2D[m, 0]), np.median(E2D[m, 1]), str(c), fontsize=11,
                fontweight='bold', ha='center', va='center',
                bbox=dict(boxstyle='circle,pad=0.25', fc='white', ec=col, lw=1.5))
    ax.set_title('Grupos encontrados', fontweight='bold')

    ax = axes[1]
    for cls in CLASSES:
        m = y == cls
        ax.scatter(E2D[m, 0], E2D[m, 1], s=28, color=CLASS_COLORS[cls], alpha=0.85,
                   edgecolor='white', linewidth=0.4, label=f'Agarre {cls}')
    ax.set_title('Agarres reales', fontweight='bold')

    for ax in axes:
        style_ax(ax)
        ax.set_xlabel(f'{AXIS} 1', fontweight='bold')
        ax.legend(fontsize=7, loc='best', frameon=True)
    axes[0].set_ylabel(f'{AXIS} 2', fontweight='bold')
    fig.suptitle(title, fontweight='bold', fontsize=11)
    plt.tight_layout()
    savefig(fig, fname)


def plot_clusters_3d(labels, title, fname):
    """Misma vista espejo que plot_clusters_2d pero en 3D (PDF estático)."""
    fig = plt.figure(figsize=(15, 6.4), dpi=300)
    fig.patch.set_facecolor('#F7F9FC')

    ax = fig.add_subplot(1, 2, 1, projection='3d')
    found = [c for c in sorted(set(labels)) if c >= 0]
    pal = sns.color_palette('Set2', max(1, len(found)))
    if (labels < 0).any():
        m = labels < 0
        ax.scatter(E3D[m, 0], E3D[m, 1], E3D[m, 2], s=12, color='#BDBDBD',
                   alpha=0.5, label='Ruido (−1)')
    for c, col in zip(found, pal):
        m = labels == c
        ax.scatter(E3D[m, 0], E3D[m, 1], E3D[m, 2], s=26, color=col, alpha=0.85,
                   edgecolor='white', linewidth=0.3, label=f'Grupo {c}')
        ax.text(np.median(E3D[m, 0]), np.median(E3D[m, 1]), np.median(E3D[m, 2]),
                str(c), fontsize=10, fontweight='bold', ha='center', va='center',
                bbox=dict(boxstyle='circle,pad=0.2', fc='white', ec=col, lw=1.2))
    ax.set_title('Grupos encontrados', fontweight='bold')

    ax = fig.add_subplot(1, 2, 2, projection='3d')
    for cls in CLASSES:
        m = y == cls
        ax.scatter(E3D[m, 0], E3D[m, 1], E3D[m, 2], s=26, color=CLASS_COLORS[cls],
                   alpha=0.85, edgecolor='white', linewidth=0.3, label=f'Agarre {cls}')
    ax.set_title('Agarres reales', fontweight='bold')

    for ax in fig.axes:
        ax.view_init(elev=30, azim=-60)
        ax.set_xlabel(f'{AXIS} 1', fontweight='bold', fontsize=8)
        ax.set_ylabel(f'{AXIS} 2', fontweight='bold', fontsize=8)
        ax.set_zlabel(f'{AXIS} 3', fontweight='bold', fontsize=8)
        ax.legend(fontsize=6, loc='best', frameon=True)
    fig.suptitle(title, fontweight='bold', fontsize=11)
    fig.tight_layout()
    savefig(fig, fname)


def plot_barrido_k(sweep_df, best_k):
    # 3 métricas en 3 paneles: Silhouette (↑), Davies-Bouldin (↓), ARI (↑).
    fig, axes = plt.subplots(1, 3, figsize=(16, 4), dpi=300)
    fig.patch.set_facecolor('#F7F9FC')

    axes[0].plot(sweep_df['k'], sweep_df['silhouette'], marker='o', color='#1E88E5')
    axes[0].set_ylabel('Silhouette', fontweight='bold')
    axes[0].set_title('Silhouette — mayor = mejor', fontweight='bold', fontsize=10)

    axes[1].plot(sweep_df['k'], sweep_df['davies_bouldin'], marker='o', color='#8E24AA')
    axes[1].set_ylabel('Davies–Bouldin', fontweight='bold')
    axes[1].set_title('Davies–Bouldin — menor = mejor', fontweight='bold', fontsize=10)

    axes[2].plot(sweep_df['k'], sweep_df['ARI_agarre'], marker='o', color='#F4511E')
    axes[2].set_ylabel('ARI', fontweight='bold')
    axes[2].set_title('ARI con agarres — mayor = mejor', fontweight='bold', fontsize=10)

    for ax in axes:
        style_ax(ax)
        ax.axvline(N_CLASSES, color='#999999', linestyle=':', linewidth=1)
        ax.axvline(best_k, color='#2E7D32', linestyle='--', linewidth=1.2)
        ax.set_xlabel('k (número de grupos)', fontweight='bold')
    plt.tight_layout()
    savefig(fig, 'kmeans_barrido_k')


def plot_gmm_bic(gmm_df, best_n):
    fig, ax = plt.subplots(figsize=(8, 4), dpi=300)
    fig.patch.set_facecolor('#F7F9FC')
    ax.plot(gmm_df['n_components'], gmm_df['BIC'], marker='o', color='#1E88E5', label='BIC')
    ax.plot(gmm_df['n_components'], gmm_df['AIC'], marker='s', color='#F4511E', label='AIC')
    ax.axvline(N_CLASSES, color='#999999', linestyle=':', linewidth=1,
               label=f'nº de agarres = {N_CLASSES}')
    ax.axvline(best_n, color='#2E7D32', linestyle='--', linewidth=1.2,
               label=f'mejor BIC = {best_n}')
    ax.set_xlabel('n_components (número de grupos)', fontweight='bold')
    ax.set_ylabel('Criterio (menor = mejor)', fontweight='bold')
    ax.set_title('Selección del número de grupos con GMM', fontweight='bold')
    ax.legend(fontsize=8)
    style_ax(ax)
    plt.tight_layout()
    savefig(fig, 'gmm_bic_aic')


def plot_agg_linkages(agg_df):
    fig, axes = plt.subplots(1, 2, figsize=(11, 4), dpi=300)
    fig.patch.set_facecolor('#F7F9FC')
    pos, width = np.arange(len(agg_df)), 0.35

    ax = axes[0]
    ax.bar(pos - width / 2, agg_df['ARI_agarre'], width, label='ARI', color='#F4511E')
    ax.bar(pos + width / 2, agg_df['silhouette'], width, label='Silhouette', color='#1E88E5')
    ax.set_xticks(pos, agg_df.index)
    ax.set_ylabel('Valor', fontweight='bold')
    ax.set_title('ARI y Silhouette — mayor = mejor', fontweight='bold', fontsize=10)
    ax.legend(fontsize=8, frameon=True)
    style_ax(ax)

    ax = axes[1]
    ax.bar(pos, agg_df['davies_bouldin'], width, label='Davies–Bouldin', color='#8E24AA')
    ax.set_xticks(pos, agg_df.index)
    ax.set_ylabel('Davies–Bouldin', fontweight='bold')
    ax.set_title('Davies–Bouldin — menor = mejor', fontweight='bold', fontsize=10)
    ax.legend(fontsize=8, frameon=True)
    style_ax(ax)

    fig.suptitle(f'Aglomerativo con k={N_CLASSES} según linkage', fontweight='bold', fontsize=11)
    plt.tight_layout()
    savefig(fig, 'aglomerativo_linkages')


def plot_azar(nulls, p_vals, tests, fname='azar_histogramas'):
    fig, axes = plt.subplots(1, len(tests), figsize=(4.6 * len(tests), 3.6), dpi=300)
    fig.patch.set_facecolor('#F7F9FC')
    for ax, t in zip(np.atleast_1d(axes), tests):
        null = nulls[t['nombre']]
        p = p_vals[t['nombre']]
        style_ax(ax)
        ax.hist(null, bins=40, color='#B0BEC5', edgecolor='white',
                label='ARI con etiquetas al azar')
        ax.axvline(t['ari'], color=t['color'], linewidth=1.6,
                   label=f"ARI real = {t['ari']:.3f}")
        ax.set_xlabel('ARI', fontweight='bold')
        ax.set_ylabel('Frecuencia', fontweight='bold')
        ax.set_title(f"{t['nombre']}\np = {p:.4f}", fontweight='bold', fontsize=10)
        ax.legend(fontsize=7)
    plt.tight_layout()
    savefig(fig, fname)


def plot_resumen_metricas(tabla, fname='resumen_metricas'):
    """Tabla-resumen en PDF: por método, las 3 métricas y el nº de grupos.
    Color: azul más intenso = mejor valor de la columna (Davies-Bouldin va invertido)."""

    columnas = [('silhouette', 'Silhouette', 'mayor = mejor'),
                ('davies_bouldin', 'Davies–Bouldin', 'menor = mejor'),
                ('ARI_agarre', 'ARI', 'mayor = mejor'),
                ('n_grupos', 'Nº grupos', 'elegido')]
    metodos = list(tabla.index)
    n_f, n_c = len(metodos), len(columnas)
    ancho_cab = 3.2                    # ancho de la columna de nombres de método
    ancho = 1.25                       # ancho de cada columna de métrica
    alto_cab = 0.95
    cmap = colormaps['Blues']

    # Normalización por columna para el color (1 = mejor); DB se invierte.
    vals = tabla[[c for c, _, _ in columnas]].astype(float)
    t_color = pd.DataFrame(np.nan, index=vals.index, columns=vals.columns)
    for c in vals.columns:
        v = vals[c]
        ok = np.isfinite(v)
        if ok.sum() == 0:
            continue
        lo, hi = v[ok].min(), v[ok].max()
        t_color.loc[ok, c] = 0.5 if hi == lo else (v[ok] - lo) / (hi - lo)
    t_color['davies_bouldin'] = 1 - t_color['davies_bouldin']
    t_color['n_grupos'] = np.nan

    fig, ax = plt.subplots(figsize=(ancho_cab + ancho * n_c + 1.5,
                                    0.62 * n_f + 2.2), dpi=300)
    fig.patch.set_facecolor('#F7F9FC')
    ax.axis('off')

    def x_col(j):
        return ancho_cab + j * ancho

    # Encabezado
    ax.add_patch(Rectangle((0, n_f), ancho_cab, alto_cab, facecolor='#37474F',
                           edgecolor='white', lw=1.5))
    ax.text(ancho_cab / 2, n_f + alto_cab / 2, 'Método', ha='center', va='center',
            fontweight='bold', fontsize=10, color='white')
    for j, (_, nombre, regla) in enumerate(columnas):
        ax.add_patch(Rectangle((x_col(j), n_f), ancho, alto_cab, facecolor='#37474F',
                               edgecolor='white', lw=1.5))
        ax.text(x_col(j) + ancho / 2, n_f + alto_cab * 0.62, nombre, ha='center',
                va='center', fontweight='bold', fontsize=9.5, color='white')
        ax.text(x_col(j) + ancho / 2, n_f + alto_cab * 0.24, f'({regla})', ha='center',
                va='center', fontsize=7, color='#B0BEC5')

    # Filas
    for i, metodo in enumerate(metodos):
        y0 = n_f - 1 - i
        ax.add_patch(Rectangle((0, y0), ancho_cab, 1, facecolor='#ECEFF1',
                               edgecolor='white', lw=1.5))
        ax.text(ancho_cab - 0.12, y0 + 0.5, metodo, ha='right', va='center',
                fontweight='bold', fontsize=9.5, color='#263238')
        for j, (col, _, _) in enumerate(columnas):
            v = vals.loc[metodo, col]
            t = t_color.loc[metodo, col]
            if pd.isna(v):
                fc, tc, txt = '#ECEFF1', '#90A4AE', '—'
            elif col == 'n_grupos':
                fc, tc, txt = '#FFF3E0', '#4E342E', str(int(v))
            else:
                txt = f'{v:.3f}'
                tt = float(t)
                fc = cmap(0.18 + 0.72 * tt)
                tc = 'white' if tt > 0.55 else '#263238'
            ax.add_patch(Rectangle((x_col(j), y0), ancho, 1, facecolor=fc,
                                   edgecolor='white', lw=1.5))
            ax.text(x_col(j) + ancho / 2, y0 + 0.5, txt, ha='center', va='center',
                    fontsize=10, color=tc, fontweight='bold')

    ax.set_xlim(0, ancho_cab + ancho * n_c)
    ax.set_ylim(-0.8, n_f + alto_cab)
    ax.set_title('Resumen de métricas de clustering por método', fontweight='bold',
                 fontsize=11, pad=12)
    fig.text(0.5, 0.035,
             'Azul más intenso = mejor valor en la columna (Davies–Bouldin va invertido, '
             'porque menor es mejor).',
             ha='center', fontsize=7.5, color='#666666')
    plt.tight_layout()
    savefig(fig, fname)


# Coordenadas 2D y 3D: se calculan una sola vez y se usan en todas las vistas
E2D, AXIS = proyectar(X, 2)
E3D, _ = proyectar(X, 3)

## 2. K-Means con k = número de agarres

Se le pide a K-Means que forme tantos grupos como agarres hay.

**Qué mirar (las 3 métricas del notebook):**
- **Silhouette:** cohesión y separación de los grupos (de −1 a 1; **mayor = mejor**).
- **Davies–Bouldin:** dispersión interna y separación relativa entre grupos (**menor = mejor**; 0 = perfecto).
- **ARI:** cuánto coinciden los grupos con los agarres reales (0 = como al azar, 1 = perfecto).
- **Tabla:** filas = grupos encontrados, columnas = agarres reales; el número es el **porcentaje (%)**
  de cada agarre que cayó en ese grupo (cada columna suma 100).
- **Vistas 2D y 3D:** izquierda coloreada por grupo, derecha por agarre real, con las mismas
  coordenadas. La proyección es solo para ver; las conclusiones salen de las métricas y de la tabla.

In [7]:
km_labels, km_res = run_kmeans(X, y, k=N_CLASSES)
display(pd.Series(km_res).round(3).to_frame('K-Means'))

km_title = (f'K-Means k={N_CLASSES} — Sil={km_res["silhouette"]:.3f} | '
            f'DB={km_res["davies_bouldin"]:.3f} | ARI={km_res["ARI_agarre"]:.3f}')
plot_contingency(km_labels, km_title, 'kmeans_tabla')
plot_clusters_2d(km_labels, km_title, 'kmeans_2d')
plot_clusters_3d(km_labels, km_title, 'kmeans_3d')

,K-Means
k,7.000
silhouette,0.166
davies_bouldin,1.565
ARI_agarre,0.175


## 3. K-Means con otros valores de k

Se repite K-Means con distintos números de grupos (k de 2 a 12), con las 3 métricas:
- **Silhouette** (izquierda): qué tan bien separados quedan los grupos; **mayor = mejor**.
  El k con el valor más alto es el número de grupos que "prefieren" los datos. Si es menor que el
  número de agarres, algunos agarres forman familias.
- **Davies–Bouldin** (centro): dispersión interna y separación relativa; **menor = mejor**.
  El mínimo marca el k con grupos más compactos y bien separados.
- **ARI** (derecha): coincidencia con los agarres para cada k (**mayor = mejor**).

In [8]:
sweep_df, best_k = barrido_kmeans(X, y)
plot_barrido_k(sweep_df, best_k)
print(f'k con mejor silhouette: {best_k}  |  número de agarres: {N_CLASSES}')

k con mejor silhouette: 2  |  número de agarres: 7


## 4. GMM (Mezcla de Gaussianas)

A diferencia de K-Means, GMM modela cada grupo como una gaussiana y asigna probabilidades en vez
de distancias duras. Se usa covarianza **diagonal** (`GMM_COVARIANCE = 'diag'`): con 150 columnas y
~600 filas, una covarianza completa tiene más de 11 000 parámetros por grupo, y el BIC terminaría
eligiendo siempre el menor número de grupos por pura penalización, no por los datos.

Se prueba `n_components` de 2 a 12 y se elige el mejor con **BIC** (menor = mejor; penaliza
modelos con más grupos para evitar sobreajustar). La línea punteada marca el número real de agarres.

**Qué mirar:** si el mínimo del BIC cae en 7, los datos "piden" 7 grupos; si cae en un valor
menor, algunos agarres forman familias indistinguibles. Si el BIC sigue bajando hasta el borde del
rango (12), no hay un mínimo claro: el BIC no es concluyente para este dataset.

In [9]:
gmm_df, best_n, gmm_labels, gmm_res = run_gmm(X, y)
plot_gmm_bic(gmm_df, best_n)
print(f'Mejor n_components según BIC: {best_n}  |  número de agarres: {N_CLASSES}')

display(pd.Series(gmm_res).round(3).to_frame('GMM'))
gmm_title = (f'GMM n={best_n} — Sil={gmm_res["silhouette"]:.3f} | '
             f'DB={gmm_res["davies_bouldin"]:.3f} | ARI={gmm_res["ARI_agarre"]:.3f}')
plot_contingency(gmm_labels, gmm_title, 'gmm_tabla')
plot_clusters_2d(gmm_labels, gmm_title, 'gmm_2d')
plot_clusters_3d(gmm_labels, gmm_title, 'gmm_3d')

Mejor n_components según BIC: 12  |  número de agarres: 7


,GMM
n_components,12.000
silhouette,0.117
davies_bouldin,1.849
ARI_agarre,0.145


## 5. Aglomerativo (jerárquico)

Construye un árbol de grupos fusionando los pares más cercanos y lo corta en `k = nº de agarres`.
Se comparan tres formas de medir la distancia entre grupos (**linkages**):

- **ward:** minimiza la varianza interna de los grupos (el más parecido a K-Means).
- **complete:** usa la distancia entre los puntos más lejanos de cada grupo.
- **average:** usa la distancia promedio.

El linkage que se muestra en detalle se elige por **Silhouette** (sin usar las etiquetas); el ARI
se usa solo para evaluar, así el p-valor de la sección 7 no queda inflado por haber escogido el
linkage que mejor coincide con los agarres.

**Qué mirar:** la figura izquierda muestra **ARI y Silhouette (mayor = mejor)** por linkage; la
derecha, **Davies–Bouldin (menor = mejor)**. Si los tres linkages dan valores parecidos, la
estructura es robusta al método; si difieren mucho, la decisión de agrupar depende del criterio.

In [10]:
agg_df, agg_labels, best_link = run_aglomerativo(X, y, k=N_CLASSES)
agg_best = agg_labels[best_link]
agg_res = agg_df.loc[best_link]
display(agg_df.round(3))
plot_agg_linkages(agg_df)

agg_title = (f'Aglomerativo ({best_link}, k={N_CLASSES}) — Sil={agg_res.silhouette:.3f} | '
             f'DB={agg_res.davies_bouldin:.3f} | ARI={agg_res.ARI_agarre:.3f}')
plot_contingency(agg_best, agg_title, 'aglomerativo_tabla')
plot_clusters_2d(agg_best, agg_title, 'aglomerativo_2d')
plot_clusters_3d(agg_best, agg_title, 'aglomerativo_3d')

,silhouette,davies_bouldin,ARI_agarre
linkage,,,
ward,0.125,1.731,0.131
complete,0.139,2.088,0.137
average,0.156,1.200,0.083


## 6. HDBSCAN con min_samples explícito

HDBSCAN decide solo cuántos grupos hay y marca como **ruido (−1)** lo que no encaja en ninguno.

**Por qué la versión anterior daba 0 grupos y 100% ruido:**

1. **`min_samples` por defecto en sklearn = `min_cluster_size`.** Con `min_cluster_size` de 15 a 40,
   cada punto necesitaba 15–40 vecinos en su radio de densidad para ser "núcleo". Con 609 filas en
   150 dimensiones eso es tan exigente que **ningún punto califica → todo es ruido → 0 grupos**.
2. **El dataset no tiene islas de densidad.** HDBSCAN busca regiones densas separadas por zonas
   raras; aquí la densidad es casi uniforme y los agarres se solapan, así que incluso bien ajustado
   encuentra pocos grupos con ARI bajo. Eso también es un resultado.

Aquí se barre `min_cluster_size` × `min_samples` con `min_samples` pequeño (1–10).
Silhouette y Davies–Bouldin se calculan **solo sobre los puntos que no son ruido**; el ARI se
calcula sobre todas las filas, con el ruido (−1) contado como un grupo más. La combinación mostrada
se elige por **Silhouette** (sin etiquetas) entre las que tienen más de un grupo y como máximo
`HDB_MAX_RUIDO` % de ruido. Un ARI alto con mucho ruido no es un buen resultado: hay que mirar ambos.

In [11]:
hdb_df, hdb_labels, best_hdb = run_hdbscan(X, y)
display(hdb_df.round(3))

if best_hdb is not None:
    r = hdb_df.loc[best_hdb]
    hdb_title = (f'HDBSCAN (mcs={best_hdb[0]}, min_samples={best_hdb[1]}) — '
                 f'{int(r.n_grupos)} grupos, {r["%_ruido"]:.0f}% ruido, '
                 f'Sil={r.silhouette:.3f}, DB={r.davies_bouldin:.3f}, ARI={r.ARI_agarre:.3f}')
    plot_contingency(hdb_labels[best_hdb], hdb_title, 'hdbscan_tabla')
    plot_clusters_2d(hdb_labels[best_hdb], hdb_title, 'hdbscan_2d')
    plot_clusters_3d(hdb_labels[best_hdb], hdb_title, 'hdbscan_3d')
else:
    print(f'HDBSCAN no encontró más de un grupo con <= {HDB_MAX_RUIDO}% de ruido en ninguna '
          'combinación: no hay regiones densas separadas en este dataset.')

n_grupos  %_ruido  silhouette  davies_bouldin  \
min_cluster_size min_samples                                                  
15               1                   3   50.903       0.186           1.120   
                 3                   0  100.000         NaN             NaN   
                 5                   0  100.000         NaN             NaN   
                 10                  0  100.000         NaN             NaN   
20               1                   2   74.713       0.287           0.956   
                 3                   0  100.000         NaN             NaN   
                 5                   0  100.000         NaN             NaN   
                 10                  0  100.000         NaN             NaN   
30               1                   0  100.000         NaN             NaN   
                 3                   0  100.000         NaN             NaN   
                 5                   0  100.000         NaN             NaN   
                 10                  0  100.000         NaN             NaN   
40               1                   0  100.000         NaN             NaN   
                 3                   0  100.000         NaN             NaN   
                 5                   0  100.000         NaN             NaN   
                 10                  0  100.000         NaN             NaN   

                              ARI_agarre  
min_cluster_size min_samples              
15               1                 0.051  
                 3                 0.000  
                 5                 0.000  
                 10                0.000  
20               1                 0.044  
                 3                 0.000  
                 5                 0.000  
                 10                0.000  
30               1                 0.000  
                 3                 0.000  
                 5                 0.000  
                 10                0.000  
40               1                 0.000  
                 3                 0.000  
                 5                 0.000  
                 10                0.000

## 7. ¿Es mejor que el azar?

Se mezclan al azar las etiquetas de agarre `N_PERM` veces y se calcula el ARI de cada algoritmo
frente a su propia distribución de azar. Si el ARI real queda muy a la derecha del histograma
(p-valor bajo), la coincidencia no es casualidad.

In [12]:
tests = [
    {'nombre': 'K-Means', 'labels': km_labels, 'ari': km_res['ARI_agarre'], 'color': '#E53935'},
    {'nombre': 'GMM', 'labels': gmm_labels, 'ari': gmm_res['ARI_agarre'], 'color': '#1E88E5'},
    {'nombre': f'Aglomerativo ({best_link})', 'labels': agg_best,
     'ari': float(agg_res.ARI_agarre), 'color': '#43A047'},
]
nulls, p_vals = prueba_azar(y, tests)
plot_azar(nulls, p_vals, tests)

## 8. Resumen

Se genera la figura **`resumen_metricas.pdf`**: una tabla con **Silhouette, Davies–Bouldin, ARI** y
el número de grupos elegido por cada método. Para comparar con otro dataset, se ejecuta este
notebook con ese dataset y se comparan las dos figuras.

In [13]:
# Tabla-resumen: 3 métricas + nº de grupos por método (PDF en SAVE_DIR)
sweep_best = sweep_df.loc[sweep_df['k'] == best_k].iloc[0]
tabla_resumen = pd.DataFrame([
    {'metodo': f'K-Means (k={N_CLASSES})', 'silhouette': km_res['silhouette'],
     'davies_bouldin': km_res['davies_bouldin'], 'ARI_agarre': km_res['ARI_agarre'],
     'n_grupos': N_CLASSES},
    {'metodo': f'K-Means (barrido, k={best_k})', 'silhouette': sweep_best['silhouette'],
     'davies_bouldin': sweep_best['davies_bouldin'],
     'ARI_agarre': sweep_best['ARI_agarre'], 'n_grupos': best_k},
    {'metodo': f'GMM (n={best_n})', 'silhouette': gmm_res['silhouette'],
     'davies_bouldin': gmm_res['davies_bouldin'], 'ARI_agarre': gmm_res['ARI_agarre'],
     'n_grupos': best_n},
    {'metodo': f'Aglomerativo ({best_link})', 'silhouette': float(agg_res.silhouette),
     'davies_bouldin': float(agg_res.davies_bouldin),
     'ARI_agarre': float(agg_res.ARI_agarre), 'n_grupos': N_CLASSES},
    {'metodo': 'HDBSCAN (mejor grid)',
     'silhouette': np.nan if best_hdb is None else hdb_df.loc[best_hdb, 'silhouette'],
     'davies_bouldin': np.nan if best_hdb is None else hdb_df.loc[best_hdb, 'davies_bouldin'],
     'ARI_agarre': np.nan if best_hdb is None else hdb_df.loc[best_hdb, 'ARI_agarre'],
     'n_grupos': np.nan if best_hdb is None else hdb_df.loc[best_hdb, 'n_grupos']},
]).set_index('metodo')
plot_resumen_metricas(tabla_resumen)
display(tabla_resumen.round(3))

,silhouette,davies_bouldin,ARI_agarre,n_grupos
metodo,,,,
K-Means (k=7),0.166,1.565,0.175,7
"K-Means (barrido, k=2)",0.263,1.294,0.079,2
GMM (n=12),0.117,1.849,0.145,12
Aglomerativo (average),0.156,1.200,0.083,7
HDBSCAN (mejor grid),0.186,1.120,0.051,3


## 9. Interpretación

*(Completar con los resultados.)*

- **¿Se forman los agarres solos?** → ARI de K-Means, GMM y aglomerativo; p-valores frente al azar.
- **¿Qué agarres se mezclan?** → tabla de K-Means/GMM; comparar con los pares más confundidos de la LDA.
- **¿Qué método da grupos más limpios?** → `resumen_metricas.pdf`: Silhouette alto,
  Davies–Bouldin bajo y ARI alto a la vez.
- **¿Cuántos grupos sugieren los datos?** → k con mejor Silhouette/menor Davies–Bouldin en el
  barrido, mínimo del BIC de GMM y nº de grupos de HDBSCAN (con `min_samples` explícito).
- **¿Hay densidades separadas?** → si HDBSCAN da ruido alto con ARI bajo, los datos no forman
  islas densas: es una propiedad del dataset, no un bug del algoritmo.

In [14]:
sello.guardar()

PosixPath('/home/Felixphz/project/sEMG-ArmBand/PROVENANCE.md')